In [1]:
import sys
from pathlib import Path

# Add parent directory (where utility.py lives) to the path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from utility import _feature_engineering, setpoint_df, FeatureCreator
from config import CONTROL_VARS
from data_cleaning import (
    unique_in_order, ordered_difference, ordered_intersection,
)

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr  # Changed import path

sns.set(rc={"figure.figsize": (12, 4)})
sns.set_style("whitegrid")
sns.set_context("notebook")

In [2]:
Y_COLUMN = "MBS_SCT_CD"

DATA_PATH = "../data/costimiser_turnup.parquet"

In [3]:
OUTPUT_VAR = "MBS_SCT_CD"
MAX_COND_DIM = 3
MIN_ABS_VAL = 0.05
ALPHA = 0.05
OUTPUT_DIR = Path("conditional_dependence_results")

In [4]:
from itertools import combinations
from pathlib import Path
from typing import Iterable, Optional

import networkx as nx
import pandas as pd
from tigramite.independence_tests.parcorr import ParCorr


class ConditionalDependenceSkeleton:
    def __init__(
        self,
        dataframe,
        *,
        output_var: str = OUTPUT_VAR,
        max_cond_dim: int = MAX_COND_DIM,
        min_abs_val: float = MIN_ABS_VAL,
        alpha: float = ALPHA,
        verbosity: int = 1,
    ):
        self.dataframe = dataframe
        self.output_var = output_var
        self.max_cond_dim = int(max_cond_dim)
        self.min_abs_val = float(min_abs_val)
        self.alpha = float(alpha)
        self.verbosity = int(verbosity)

        self.var_names = list(dataframe.var_names)
        self.name_to_idx = {
            name: i for i, name in enumerate(self.var_names)
        }

        if self.output_var not in self.name_to_idx:
            raise ValueError(
                f"Output variable {self.output_var!r} "
                "is not in dataframe.var_names"
            )

        self.triple_parcorr = ParCorr(significance="analytic")
        self.triple_parcorr.set_dataframe(self.dataframe)

        self.graph_: Optional[nx.Graph] = None
        self.separation_df_: Optional[pd.DataFrame] = None
        self.edge_diagnostics_df_: Optional[pd.DataFrame] = None

        self.target_screening_df_: Optional[pd.DataFrame] = None
        self.separator_motifs_df_: Optional[pd.DataFrame] = None
        self.selected_variables_: Optional[list[str]] = None
        self.separating_variables_: Optional[list[str]] = None

    # ============================================================
    # Basic CI utilities
    # ============================================================

    def parcorr_test(
        self,
        x_name: str,
        y_name: str,
        z_names: Optional[Iterable[str]] = None,
    ) -> dict:
        if z_names is None:
            z_names = []

        z_names = list(z_names)

        x = self.name_to_idx[x_name]
        y = self.name_to_idx[y_name]
        z = [self.name_to_idx[name] for name in z_names]

        val, pval = self.triple_parcorr.run_test(
            X=[(x, 0)],
            Y=[(y, 0)],
            Z=[(zz, 0) for zz in z],
        )

        return {
            "val": float(val),
            "p": float(pval),
        }

    def classify_test(self, result: dict) -> str:
        val = result["val"]
        p = result["p"]

        if p < self.alpha:
            if abs(val) >= self.min_abs_val:
                return "DEP"
            return "WEAK"

        return "INDEP"

    def practically_independent(self, result: dict) -> bool:
        return abs(result["val"]) < self.min_abs_val

    @staticmethod
    def _complete_graph(variable_names: list[str]) -> nx.Graph:
        G = nx.Graph()
        G.add_nodes_from(variable_names)
        G.add_edges_from(combinations(variable_names, 2))
        return G

    # ============================================================
    # PHASE 1
    # Target screening
    # ============================================================

    def _target_screening(
        self,
        variable_names: list[str],
    ):
        target = self.output_var

        all_candidates = [
            v
            for v in variable_names
            if v != target
        ]

        diagnostics = []
        separation_records = []
        motif_records = []

        if self.verbosity:
            print("\n" + "=" * 70)
            print("PHASE 1: TARGET SCREENING")
            print("=" * 70)
            print(f"Target: {target}")
            print(f"Candidate variables: {len(all_candidates):,}")

        # --------------------------------------------------------
        # k = 0
        # --------------------------------------------------------

        if self.verbosity:
            print("\nTarget screening: conditioning dimension 0...")

        marginal_survivors = []

        for x_name in all_candidates:
            result = self.parcorr_test(
                target,
                x_name,
                [],
            )

            abs_val = abs(result["val"])

            if self.practically_independent(result):
                separation_records.append({
                    "X": x_name,
                    "Y": target,
                    "conditioning_set": tuple(),
                    "cond_dim": 0,
                    "val": result["val"],
                    "abs_val": abs_val,
                    "p": result["p"],
                    "status": self.classify_test(result),
                })

                diagnostics.append({
                    "X": x_name,
                    "Y": target,
                    "marginal_val": result["val"],
                    "marginal_abs_val": abs_val,
                    "marginal_p": result["p"],
                    "best_separator": None,
                    "conditional_val": None,
                    "conditional_abs_val": None,
                    "conditional_p": None,
                    "retained": False,
                })

            else:
                marginal_survivors.append(x_name)

                diagnostics.append({
                    "X": x_name,
                    "Y": target,
                    "marginal_val": result["val"],
                    "marginal_abs_val": abs_val,
                    "marginal_p": result["p"],
                    "best_separator": None,
                    "conditional_val": None,
                    "conditional_abs_val": None,
                    "conditional_p": None,
                    "retained": True,
                })

        if self.verbosity:
            print(
                f"Marginally associated with {target}: "
                f"{len(marginal_survivors):,}"
            )

        # --------------------------------------------------------
        # k = 1
        # Search ALL Z and keep the best separator for each X
        # --------------------------------------------------------

        robust_neighbours = []
        separating_variables = set()

        if self.max_cond_dim >= 1:

            if self.verbosity:
                print(
                    "\nTarget screening: conditioning dimension 1..."
                )
                print(
                    "Testing all single-variable conditioners "
                    "from the full variable pool."
                )

            diag_lookup = {
                row["X"]: row
                for row in diagnostics
            }

            total_tested = 0

            for i, x_name in enumerate(
                marginal_survivors,
                start=1,
            ):

                candidate_conditioners = [
                    z
                    for z in all_candidates
                    if z != x_name
                ]

                best_separator = None
                best_result = None
                best_abs_val = float("inf")

                for z_name in candidate_conditioners:

                    result = self.parcorr_test(
                        target,
                        x_name,
                        [z_name],
                    )

                    total_tested += 1
                    abs_val = abs(result["val"])

                    if abs_val < best_abs_val:
                        best_abs_val = abs_val
                        best_separator = z_name
                        best_result = result

                diag = diag_lookup[x_name]

                if best_result is not None:
                    diag["best_separator"] = best_separator
                    diag["conditional_val"] = best_result["val"]
                    diag["conditional_abs_val"] = best_abs_val
                    diag["conditional_p"] = best_result["p"]

                if (
                    best_result is not None
                    and best_abs_val < self.min_abs_val
                ):
                    separating_variables.add(best_separator)

                    separation_records.append({
                        "X": x_name,
                        "Y": target,
                        "conditioning_set": (best_separator,),
                        "cond_dim": 1,
                        "val": best_result["val"],
                        "abs_val": best_abs_val,
                        "p": best_result["p"],
                        "status": self.classify_test(best_result),
                    })

                    motif_records.append({
                        "associated_variable": x_name,
                        "separator": best_separator,
                        "target": target,
                        "marginal_val": diag["marginal_val"],
                        "marginal_abs_val": diag["marginal_abs_val"],
                        "marginal_p": diag["marginal_p"],
                        "conditional_val": best_result["val"],
                        "conditional_abs_val": best_abs_val,
                        "conditional_p": best_result["p"],
                        "attenuation": (
                            1.0
                            - best_abs_val
                            / diag["marginal_abs_val"]
                            if diag["marginal_abs_val"] > 0
                            else None
                        ),
                    })

                    diag["retained"] = False

                else:
                    robust_neighbours.append(x_name)
                    diag["retained"] = True

                if self.verbosity >= 2 and i % 25 == 0:
                    print(
                        f"Processed {i:,}/"
                        f"{len(marginal_survivors):,} candidates..."
                    )

            if self.verbosity:
                print(f"Tested {total_tested:,} CI relations")
                print(
                    f"Robust target neighbours: "
                    f"{len(robust_neighbours):,}"
                )
                print(
                    f"Distinct best separators: "
                    f"{len(separating_variables):,}"
                )

        else:
            robust_neighbours = list(marginal_survivors)

        self.target_screening_df_ = pd.DataFrame(diagnostics)
        self.separator_motifs_df_ = pd.DataFrame(motif_records)
        self.separating_variables_ = sorted(separating_variables)

        return (
            robust_neighbours,
            separation_records,
        )

    # ============================================================
    # PHASE 2
    # Deeper skeleton only among robust target neighbours
    # ============================================================

    def _fit_robust_skeleton(
        self,
        variable_names: list[str],
    ):
        G = self._complete_graph(variable_names)

        separation_records = []
        best_by_edge = {}

        for k in range(self.max_cond_dim + 1):

            if self.verbosity:
                print(
                    f"\nRobust skeleton: conditioning sets "
                    f"of size {k}..."
                )

            adjacency_snapshot = {
                node: set(G.neighbors(node))
                for node in G.nodes
            }

            edges_snapshot = list(G.edges())

            tested_this_round = 0
            removed_this_round = 0

            for x_name, y_name in edges_snapshot:

                if not G.has_edge(x_name, y_name):
                    continue

                if k == 0:
                    cond_sets = [tuple()]

                else:
                    neigh_x = (
                        adjacency_snapshot[x_name]
                        - {y_name}
                    )

                    neigh_y = (
                        adjacency_snapshot[y_name]
                        - {x_name}
                    )

                    candidates = sorted(
                        neigh_x | neigh_y
                    )

                    if len(candidates) < k:
                        continue

                    cond_sets = combinations(
                        candidates,
                        k,
                    )

                for cond_set in cond_sets:

                    tested_this_round += 1

                    result = self.parcorr_test(
                        x_name,
                        y_name,
                        cond_set,
                    )

                    abs_val = abs(result["val"])

                    edge_key = tuple(
                        sorted((x_name, y_name))
                    )

                    current_best = best_by_edge.get(
                        edge_key
                    )

                    if (
                        current_best is None
                        or abs_val < current_best["abs_val"]
                    ):
                        best_by_edge[edge_key] = {
                            "X": edge_key[0],
                            "Y": edge_key[1],
                            "best_conditioning_set":
                                tuple(cond_set),
                            "best_cond_dim": k,
                            "val": result["val"],
                            "abs_val": abs_val,
                            "p": result["p"],
                            "status":
                                self.classify_test(result),
                        }

                    if self.practically_independent(result):

                        G.remove_edge(
                            x_name,
                            y_name,
                        )

                        separation_records.append({
                            "X": edge_key[0],
                            "Y": edge_key[1],
                            "conditioning_set":
                                tuple(cond_set),
                            "cond_dim": k,
                            "val": result["val"],
                            "abs_val": abs_val,
                            "p": result["p"],
                            "status":
                                self.classify_test(result),
                        })

                        removed_this_round += 1
                        break

            if self.verbosity:
                print(
                    f"Tested {tested_this_round:,} CI relations"
                )
                print(
                    f"Removed {removed_this_round:,} edges; "
                    f"{G.number_of_edges():,} remain"
                )

            next_k = k + 1

            if next_k <= self.max_cond_dim:

                can_continue = False

                for x_name, y_name in G.edges():

                    candidates = (
                        (
                            set(G.neighbors(x_name))
                            - {y_name}
                        )
                        |
                        (
                            set(G.neighbors(y_name))
                            - {x_name}
                        )
                    )

                    if len(candidates) >= next_k:
                        can_continue = True
                        break

                if not can_continue:

                    if self.verbosity:
                        print(
                            "Stopping early: no remaining edge "
                            "has enough candidate conditioners "
                            f"for dimension {next_k}."
                        )

                    break

        return (
            G,
            separation_records,
            best_by_edge,
        )

    # ============================================================
    # Main fit
    # ============================================================

    def fit(
        self,
        variable_names: Optional[Iterable[str]] = None,
    ) -> nx.Graph:

        if variable_names is None:
            variable_names = list(self.var_names)
        else:
            variable_names = list(variable_names)

        missing = [
            v
            for v in variable_names
            if v not in self.name_to_idx
        ]

        if missing:
            raise ValueError(
                f"Variables not found in dataframe: {missing}"
            )

        if self.output_var not in variable_names:
            variable_names = [
                self.output_var,
                *variable_names,
            ]

        if self.verbosity:
            print(
                f"Variables entering screening: "
                f"{len(variable_names):,}"
            )

        # ========================================================
        # Phase 1
        # ========================================================

        (
            robust_neighbours,
            target_separation_records,
        ) = self._target_screening(
            variable_names
        )

        # ========================================================
        # Phase 2 variable set
        # ========================================================

        selected_variables = [
            self.output_var,
            *robust_neighbours,
        ]

        self.selected_variables_ = selected_variables

        if self.verbosity:
            print("\n" + "=" * 70)
            print("PHASE 2: ROBUST TARGET SKELETON")
            print("=" * 70)
            print(
                f"Variables entering deeper skeleton: "
                f"{len(selected_variables):,}"
            )

        # ========================================================
        # Phase 2
        # ========================================================

        (
            G,
            reduced_separation_records,
            best_by_edge,
        ) = self._fit_robust_skeleton(
            selected_variables
        )

        self.graph_ = G

        self.separation_df_ = pd.DataFrame(
            target_separation_records
            + reduced_separation_records,
            columns=[
                "X",
                "Y",
                "conditioning_set",
                "cond_dim",
                "val",
                "abs_val",
                "p",
                "status",
            ],
        )

        self.edge_diagnostics_df_ = pd.DataFrame(
            list(best_by_edge.values()),
            columns=[
                "X",
                "Y",
                "best_conditioning_set",
                "best_cond_dim",
                "val",
                "abs_val",
                "p",
                "status",
            ],
        )

        if not self.edge_diagnostics_df_.empty:

            retained = {
                tuple(sorted(edge))
                for edge in G.edges()
            }

            self.edge_diagnostics_df_["retained"] = [
                (row.X, row.Y) in retained
                for row
                in self.edge_diagnostics_df_.itertuples()
            ]

        return G

    # ============================================================
    # Helpers
    # ============================================================

    def neighbours(self, variable: str) -> list[str]:

        if self.graph_ is None:
            raise RuntimeError("Call fit() first.")

        return sorted(
            self.graph_.neighbors(variable)
        )

    def separating_record(
        self,
        x_name: str,
        y_name: str,
    ) -> pd.DataFrame:

        if self.separation_df_ is None:
            raise RuntimeError("Call fit() first.")

        a, b = sorted((x_name, y_name))

        temp = self.separation_df_.copy()

        mask = temp.apply(
            lambda row:
                tuple(sorted((row["X"], row["Y"])))
                == (a, b),
            axis=1,
        )

        return temp.loc[mask].copy()

    def local_subgraph(
        self,
        center: Optional[str] = None,
        depth: int = 2,
    ) -> nx.Graph:

        if self.graph_ is None:
            raise RuntimeError("Call fit() first.")

        if center is None:
            center = self.output_var

        nodes = {center}
        frontier = {center}

        for _ in range(depth):

            new_frontier = set()

            for node in frontier:
                new_frontier.update(
                    self.graph_.neighbors(node)
                )

            new_frontier -= nodes

            if not new_frontier:
                break

            nodes.update(new_frontier)
            frontier = new_frontier

        return self.graph_.subgraph(nodes).copy()

    def plot_edge_table(
        self,
        graph: Optional[nx.Graph] = None,
    ) -> pd.DataFrame:

        if graph is None:

            if self.graph_ is None:
                raise RuntimeError("Call fit() first.")

            graph = self.graph_

        rows = []

        for a, b in graph.edges():

            if a == self.output_var:

                rows.append({
                    "source": b,
                    "target": a,
                    "type": "to_output",
                })

            elif b == self.output_var:

                rows.append({
                    "source": a,
                    "target": b,
                    "type": "to_output",
                })

            else:

                aa, bb = sorted((a, b))

                rows.append({
                    "source": aa,
                    "target": bb,
                    "type": "undirected",
                })

        return pd.DataFrame(rows)

    def save_results(
        self,
        output_dir: Path | str = OUTPUT_DIR,
        local_depth: int = 2,
    ) -> None:

        if self.graph_ is None:
            raise RuntimeError("Call fit() first.")

        output_dir = Path(output_dir)
        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        sep = self.separation_df_.copy()

        if not sep.empty:

            sep["conditioning_set"] = (
                sep["conditioning_set"].apply(
                    lambda x: " | ".join(x)
                )
            )

            sep.to_csv(
                output_dir / "separation_sets.csv",
                index=False,
            )

        diag = self.edge_diagnostics_df_.copy()

        if not diag.empty:

            diag["best_conditioning_set"] = (
                diag["best_conditioning_set"].apply(
                    lambda x: " | ".join(x)
                )
            )

            diag.to_csv(
                output_dir / "edge_diagnostics.csv",
                index=False,
            )

        if (
            self.target_screening_df_ is not None
            and not self.target_screening_df_.empty
        ):
            self.target_screening_df_.to_csv(
                output_dir / "target_screening.csv",
                index=False,
            )

        if (
            self.separator_motifs_df_ is not None
            and not self.separator_motifs_df_.empty
        ):
            self.separator_motifs_df_.to_csv(
                output_dir / "separator_motifs.csv",
                index=False,
            )

        if self.selected_variables_ is not None:

            pd.DataFrame({
                "variable": self.selected_variables_
            }).to_csv(
                output_dir / "selected_variables.csv",
                index=False,
            )

        if self.separating_variables_ is not None:

            pd.DataFrame({
                "variable": self.separating_variables_
            }).to_csv(
                output_dir / "separating_variables.csv",
                index=False,
            )

        nx.write_graphml(
            self.graph_,
            output_dir / "dependence_skeleton.graphml",
        )

        local_graph = self.local_subgraph(
            self.output_var,
            depth=local_depth,
        )

        nx.write_graphml(
            local_graph,
            output_dir
            / f"{self.output_var}_local.graphml",
        )

        self.plot_edge_table(
            local_graph
        ).to_csv(
            output_dir
            / f"{self.output_var}_local_edges.csv",
            index=False,
        )

In [5]:
def run_dependence_analysis(
    dataframe,
    *,
    output_var: str = OUTPUT_VAR,
    variable_names: Optional[Iterable[str]] = None,
    max_cond_dim: int = MAX_COND_DIM,
    min_abs_val: float = MIN_ABS_VAL,
    alpha: float = ALPHA,
    output_dir: Optional[Path | str] = OUTPUT_DIR,
    verbosity: int = 1,
    max_target_neighbours: int = 40,
):
    analyser = ConditionalDependenceSkeleton(
        dataframe=dataframe,
        output_var=output_var,
        max_cond_dim=max_cond_dim,
        min_abs_val=min_abs_val,
        alpha=alpha,
        verbosity=verbosity,
    )

    graph = analyser.fit(variable_names=variable_names)
    output_neighbours = analyser.neighbours(output_var)

    if verbosity:
        print("\n" + "=" * 70)
        print("FINAL RESULT")
        print("=" * 70)
        print(f"Nodes: {graph.number_of_nodes()}")
        print(f"Edges: {graph.number_of_edges()}")
        print(f"\nRemaining direct neighbours of {output_var}:")
        for name in output_neighbours:
            print(f"  - {name}")

    if output_dir is not None:
        analyser.save_results(output_dir)
        if verbosity:
            print(f"\nResults written to: {Path(output_dir).resolve()}")

    local_graph = analyser.local_subgraph(output_var, depth=2)

    return {
        "analyser": analyser,
        "graph": graph,
        "separation_df": analyser.separation_df_,
        "edge_diagnostics_df": analyser.edge_diagnostics_df_,
        "output_neighbours": output_neighbours,
        "local_graph": local_graph,
        "local_edges": analyser.plot_edge_table(local_graph),
    }


In [8]:
def ewm_filter_by_segment(
    df,
    columns,
    halflife="120min",
    max_gap="12h",
):
    df = df.copy().sort_index()

    # New segment at grade change or >12 h gap
    grade_change = (
        df["AB_Grade_ID"]
        .ne(df["AB_Grade_ID"].shift().fillna(True))
    )

    time_gap = (
        df.index.to_series()
        .diff()
        .gt(pd.Timedelta(max_gap)
        ).fillna(False)
    )

    segment_start = (grade_change | time_gap).fillna(True)

    segment = pd.Series(
        segment_start.to_numpy(dtype=bool),
        index=df.index,
    ).cumsum()

    result = df.copy()

    for _, idx in df.groupby(segment).groups.items():

        block = df.loc[idx, columns]

        result.loc[idx, columns] = (
            block
            .ewm(
                halflife=halflife,
                times=block.index,
                adjust=True,
            )
            .mean()
        )

    return result

In [15]:
[v for v in turnup_data.columns if "specific" in v.lower()]

['Specific_Starch_Uptake_Bottom_Roll__kg/T',
 'Specific_Starch_Uptake_Top_Roll__kg/T',
 'Electricity_Specific__kWh/T_',
 'Electricity_Specific__€/T_']

In [6]:
import pandas as pd
import tigramite.data_processing as pp


# ============================================================
# 1. Load raw data
# ============================================================

turnup_data = pd.read_parquet(DATA_PATH)


# ============================================================
# 2. Build variable list
# ============================================================


created_vars = []


# ============================================================
# 3. Feature engineering
# ============================================================

turnup_data = _feature_engineering(
    turnup_data,
    setpoint_df,
    steam_null=False,
    clip=True,
)

ctl_vars = unique_in_order(
    v for v in turnup_data.columns
    if "mbs" not in v.lower() and
    "wedge" not in v.lower() and
    "m2" not in v.lower() and
    "€" not in v and
    "grade" not in v.lower() and
    "overprocessing" not in v.lower() and
    "underprocessing" not in v.lower() and 
    "grammage_check" not in v.lower() and
    "paper_type" not in v.lower()
)


turnup_data = (
    turnup_data
    .set_index("Wedge_Time")
    .sort_index()
)


if len(created_vars) > 0:

    fc = FeatureCreator(
        features_to_create=created_vars,
        features_to_keep=list(
            set(
                turnup_data.columns.to_list()
                + created_vars
            )
        ),
    )

    turnup_data = fc.fit_transform(turnup_data)

    fr = fc.features_required()

else:
    fr = []



var_names = [Y_COLUMN]  + ctl_vars


# ============================================================
# 4. Create delta basis weight
# ============================================================
#
# This is created from the ORIGINAL, non-EWM-filtered values.
#
# Ideally:
#
#   actual basis weight - target basis weight
#
# Here grammage is assumed to represent the intended/nominal
# basis weight.
# ============================================================

turnup_data["delta_basis_weight"] = (
    turnup_data["Current_basis_weight"]
    - turnup_data["grammage"]
)



# ============================================================
# 5. Define variables for the within-grade analysis
# ============================================================
#
# Remove absolute basis weight and grammage.
# Replace them with delta_basis_weight.
# ============================================================

within_product_vars = [
    v
    for v in var_names
    if v not in {
        "Current_basis_weight",
        "grammage",
        "retention"
    }
]


within_product_vars = unique_in_order(
    within_product_vars
)


# ============================================================
# 6. Within-grade transformation
# ============================================================
#
# IMPORTANT:
# This is done BEFORE EWM filtering.
#
# For each variable:
#
#   X_within = X - mean(X | grade)
#
# Therefore all variables become deviations from their
# grade-specific mean.
#
# Example:
#
#   SCT_within =
#       SCT - average SCT for that grade
#
# ============================================================

GRADE_VAR = "AB_Grade_ID"

turnup_within = turnup_data.copy()


grade_means = (
    turnup_data
    .groupby(GRADE_VAR)[within_product_vars]
    .transform("mean")
)


turnup_within[within_product_vars] = (
    turnup_data[within_product_vars]
    - grade_means
)


# ============================================================
# 7. Optional sanity checks
# ============================================================

print(
    turnup_within
    .groupby(GRADE_VAR)[within_product_vars]
    .mean()
    .round(6)
)

c:\workspace\costimiser\utility.py:2845: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  turnup = turnup.assign(
c:\workspace\costimiser\utility.py:2845: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  turnup = turnup.assign(
c:\workspace\costimiser\utility.py:7280: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.co

             MBS_SCT_CD  Stock_pump_power     Speed  Current_reel_weight  \
AB_Grade_ID                                                                
3200115             0.0          0.000028 -0.000040             0.000886   
3300085             0.0          0.000023  0.000076             0.000703   
3300100             0.0          0.000006 -0.000042            -0.002209   
3300110            -0.0         -0.000034  0.000047            -0.000781   
3300120             0.0         -0.000022  0.000048             0.000000   
3300125             0.0          0.000000  0.000000            -0.001953   
6010085             0.0          0.000031 -0.000033             0.001625   
6010090            -0.0         -0.000002  0.000073             0.000657   
6010100             0.0         -0.000024 -0.000071            -0.002131   
6010110             0.0          0.000005 -0.000009            -0.001502   
6010120            -0.0         -0.000011 -0.000026             0.002643   

           

C:\Users\Gbmiggon\AppData\Local\Temp\ipykernel_42900\4058886440.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  turnup_data["delta_basis_weight"] = (


In [9]:
turnup_within[within_product_vars] = (
    turnup_within[within_product_vars]
    .astype("float64")
)

filtered_within = ewm_filter_by_segment(
    turnup_within,
    columns=within_product_vars,
    halflife="240min",
)

analysis_df = (
    filtered_within[within_product_vars]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

data_array = analysis_df.to_numpy(dtype=float)

dataframe_within = pp.DataFrame(
    data_array,
    var_names=within_product_vars,
)

# ============================================================
# 10. Conditional-dependence skeleton
# ============================================================

results_within = run_dependence_analysis(
    dataframe=dataframe_within,
    output_var=Y_COLUMN,
    max_cond_dim=3,
    min_abs_val=0.05,
    max_target_neighbours=40,
)

Variables entering screening: 499

PHASE 1: TARGET SCREENING
Target: MBS_SCT_CD
Candidate variables: 498

Target screening: conditioning dimension 0...
Marginally associated with MBS_SCT_CD: 281

Target screening: conditioning dimension 1...
Testing all single-variable conditioners from the full variable pool.
Tested 139,657 CI relations
Robust target neighbours: 28
Distinct best separators: 109

PHASE 2: ROBUST TARGET SKELETON
Variables entering deeper skeleton: 29

Robust skeleton: conditioning sets of size 0...
Tested 406 CI relations
Removed 107 edges; 299 remain

Robust skeleton: conditioning sets of size 1...
Tested 5,277 CI relations
Removed 140 edges; 159 remain

Robust skeleton: conditioning sets of size 2...
Tested 15,894 CI relations
Removed 65 edges; 94 remain

Robust skeleton: conditioning sets of size 3...
Tested 14,248 CI relations
Removed 23 edges; 71 remain

FINAL RESULT
Nodes: 29
Edges: 71

Remaining direct neighbours of MBS_SCT_CD:
  - Current_reel_dry_average
  - Dr

In [11]:
import json
from pathlib import Path

import pandas as pd


def plot_interactive_dependency_graph(
    results,
    output_var="MBS_SCT_CD",
    output_label="SCT CD",
    output_file="interactive_dependence_graph.html",
    height="850px",
    show_blocked_relations=True,
):
    """
    Create an interactive dependence graph.

    Rules
    -----
    1. Output is always visible.
    2. Robust neighbours are always visible.
    3. A separator is initially visible only if it is also a robust neighbour.
    4. Pure associated/blocked variables start hidden.
    5. Clicking a visible separator expands/collapses its associated variables.
    6. If show_blocked_relations=True, revealed blocked variables also get
       dashed X -> output relations.
    7. No pure separator is shown as an orphan in the collapsed graph.

    Parameters
    ----------
    results : dict
        Result dictionary from run_dependence_analysis().

    output_var : str
        Name of output variable.

    output_label : str
        Display label for output node.

    output_file : str or Path
        HTML output path.

    height : str
        HTML graph height.

    show_blocked_relations : bool
        If True, revealed blocked variables also show dashed X -> output links.

    Returns
    -------
    Path
        Path to generated HTML.
    """

    # ============================================================
    # Retrieve motif dataframe
    # ============================================================

    if "separator_motifs_df" in results:
        motifs = results["separator_motifs_df"].copy()

    else:
        analyser = results.get("analyser")

        if analyser is None:
            raise KeyError(
                "Could not find 'separator_motifs_df' or 'analyser' in results."
            )

        if not hasattr(analyser, "separator_motifs_df_"):
            raise AttributeError(
                "The analyser does not contain 'separator_motifs_df_'."
            )

        motifs = analyser.separator_motifs_df_.copy()

    # ============================================================
    # Robust neighbours
    # ============================================================

    if "output_neighbours" not in results:
        raise KeyError(
            "results does not contain 'output_neighbours'."
        )

    robust = [
        str(x)
        for x in results["output_neighbours"]
    ]

    output_var = str(output_var)

    # ============================================================
    # Validate motifs
    # ============================================================

    required_columns = {
        "associated_variable",
        "separator",
        "marginal_abs_val",
        "conditional_abs_val",
        "attenuation",
    }

    missing = (
        required_columns
        - set(motifs.columns)
    )

    if missing:
        raise ValueError(
            "separator_motifs_df is missing columns: "
            f"{sorted(missing)}"
        )

    motifs = (
        motifs
        .dropna(
            subset=[
                "associated_variable",
                "separator",
            ]
        )
        .copy()
    )

    motifs["associated_variable"] = (
        motifs["associated_variable"]
        .astype(str)
    )

    motifs["separator"] = (
        motifs["separator"]
        .astype(str)
    )

    # ============================================================
    # Roles
    # ============================================================

    separators = sorted(
        motifs["separator"]
        .drop_duplicates()
        .tolist()
    )

    associated = sorted(
        motifs["associated_variable"]
        .drop_duplicates()
        .tolist()
    )

    robust_set = set(robust)
    separator_set = set(separators)
    associated_set = set(associated)

    # ============================================================
    # Only initially meaningful nodes are included visibly
    #
    # Pure separators and pure associated vars still exist in the
    # dataset, but start hidden.
    # ============================================================

    all_names = (
        [output_var]
        + robust
        + separators
        + associated
    )

    all_names = list(
        dict.fromkeys(all_names)
    )

    node_id = {
        name: f"n{i}"
        for i, name in enumerate(all_names)
    }

    # ============================================================
    # Nodes
    # ============================================================

    nodes = []

    for name in all_names:

        # --------------------------------------------------------
        # Output
        # --------------------------------------------------------

        if name == output_var:

            nodes.append({
                "id": node_id[name],
                "label": output_label,
                "title": (
                    "<b>Output</b><br>"
                    + output_var
                ),
                "group": "output",
                "level": 3,
                "shape": "ellipse",
                "size": 32,
                "hidden": False,
            })

            continue

        # --------------------------------------------------------
        # Robust neighbour
        # --------------------------------------------------------

        if name in robust_set:

            title = (
                "<b>Robust SCT neighbour</b><br>"
                + name
            )

            if name in separator_set:

                n_children = int(
                    motifs["separator"]
                    .eq(name)
                    .sum()
                )

                title += (
                    "<br><br>"
                    "<b>Also acts as separator</b><br>"
                    f"{n_children} blocked variable"
                    f"{'' if n_children == 1 else 's'}"
                    "<br><br>"
                    "<b>Click to expand/collapse</b>"
                )

            nodes.append({
                "id": node_id[name],
                "label": name,
                "title": title,
                "group": "robust",
                "level": 2,
                "shape": "dot",
                "size": 24,
                "hidden": False,
            })

            continue

        # --------------------------------------------------------
        # Pure separator
        #
        # Important:
        # hidden initially, otherwise it becomes an orphan.
        # --------------------------------------------------------

        if name in separator_set:

            n_children = int(
                motifs["separator"]
                .eq(name)
                .sum()
            )

            title = (
                "<b>Separating variable</b><br>"
                + name
                + "<br><br>"
                + f"{n_children} blocked variable"
                + (
                    ""
                    if n_children == 1
                    else "s"
                )
            )

            nodes.append({
                "id": node_id[name],
                "label": name,
                "title": title,
                "group": "separator",
                "level": 1,
                "shape": "dot",
                "size": 18,
                "hidden": True,
            })

            continue

        # --------------------------------------------------------
        # Pure associated / blocked variable
        # --------------------------------------------------------

        if name in associated_set:

            rows = motifs[
                motifs["associated_variable"]
                == name
            ]

            row = rows.iloc[0]

            marginal_abs_val = row["marginal_abs_val"]
            conditional_abs_val = row["conditional_abs_val"]
            attenuation = row["attenuation"]
            separator_name = str(row["separator"])

            marginal_txt = (
                f"{marginal_abs_val:.3f}"
                if pd.notna(marginal_abs_val)
                else "NA"
            )

            conditional_txt = (
                f"{conditional_abs_val:.3f}"
                if pd.notna(conditional_abs_val)
                else "NA"
            )

            attenuation_txt = (
                f"{100 * attenuation:.1f}%"
                if pd.notna(attenuation)
                else "NA"
            )

            title = (
                "Blocked process variable\n"
                + name
                + "\n\n"
                + "Blocking conditioning set:\n"
                + separator_name
                + "\n\n"
                + f"|r marginal| = {marginal_txt}\n"
                + f"|r conditioned| = {conditional_txt}\n"
                + f"Attenuation = {attenuation_txt}"
            )

            nodes.append({
                "id": node_id[name],
                "label": name,
                "title": title,
                "group": "associated",
                "level": 0,
                "shape": "dot",
                "size": 11,
                "hidden": True,
            })

    # ============================================================
    # Duplicate-ID guard
    # ============================================================

    node_ids = [
        n["id"]
        for n in nodes
    ]

    if len(node_ids) != len(set(node_ids)):
        raise RuntimeError(
            "Duplicate node IDs detected."
        )

    # ============================================================
    # Edges
    # ============================================================

    edges = []
    edge_counter = 0

    # ------------------------------------------------------------
    # Robust -> output
    # ------------------------------------------------------------

    for name in robust:

        if name == output_var:
            continue

        if name not in node_id:
            continue

        edge_counter += 1

        edges.append({
            "id": f"e{edge_counter}",
            "from": node_id[name],
            "to": node_id[output_var],
            "arrows": "to",
            "width": 2.5,
            "dashes": False,
            "type": "robust",
            "hidden": False,
        })

    # ------------------------------------------------------------
    # Associated -> separator
    #
    # Always available, initially hidden.
    # ------------------------------------------------------------

    for _, row in motifs.iterrows():

        x = str(row["associated_variable"])
        z = str(row["separator"])

        if (
            x not in node_id
            or z not in node_id
        ):
            continue

        edge_counter += 1

        edges.append({
            "id": f"e{edge_counter}",
            "from": node_id[x],
            "to": node_id[z],
            "width": 1.5,
            "dashes": False,
            "type": "motif",
            "hidden": True,
        })

    # ------------------------------------------------------------
    # Optional dashed X -> output
    #
    # Initially hidden. They appear only when X is revealed.
    # ------------------------------------------------------------

    if show_blocked_relations:

        added = set()

        for _, row in motifs.iterrows():

            x = str(row["associated_variable"])

            if x == output_var:
                continue

            if x in robust_set:
                continue

            if x in added:
                continue

            if x not in node_id:
                continue

            added.add(x)

            edge_counter += 1

            edges.append({
                "id": f"e{edge_counter}",
                "from": node_id[x],
                "to": node_id[output_var],
                "arrows": "to",
                "width": 1.1,
                "dashes": True,
                "type": "blocked",
                "hidden": True,
            })

    # ============================================================
    # Separator -> children mapping
    # ============================================================

    children_by_separator = {}

    for separator, group in motifs.groupby(
        "separator",
        sort=False,
    ):

        separator = str(separator)

        if separator not in node_id:
            continue

        children = [
            node_id[str(x)]
            for x in (
                group["associated_variable"]
                .drop_duplicates()
                .tolist()
            )
            if str(x) in node_id
        ]

        children_by_separator[
            node_id[separator]
        ] = children

    # ============================================================
    # Reverse mapping:
    # child -> separators
    # ============================================================

    separators_by_child = {}

    for separator_id, children in children_by_separator.items():

        for child_id in children:

            separators_by_child.setdefault(
                child_id,
                []
            ).append(
                separator_id
            )

    # ============================================================
    # Pure associated nodes
    # ============================================================

    collapsible_nodes = {
        node_id[name]
        for name in associated
        if (
            name not in robust_set
            and name not in separator_set
            and name != output_var
        )
    }

    # ============================================================
    # Pure separators
    # ============================================================

    pure_separator_nodes = {
        node_id[name]
        for name in separators
        if (
            name not in robust_set
            and name != output_var
        )
    }

    # ============================================================
    # JSON
    # ============================================================

    nodes_json = json.dumps(
        nodes,
        ensure_ascii=False,
    )

    edges_json = json.dumps(
        edges,
        ensure_ascii=False,
    )

    children_json = json.dumps(
        children_by_separator,
        ensure_ascii=False,
    )

    separators_by_child_json = json.dumps(
        separators_by_child,
        ensure_ascii=False,
    )

    collapsible_json = json.dumps(
        list(collapsible_nodes),
        ensure_ascii=False,
    )

    pure_separator_json = json.dumps(
        list(pure_separator_nodes),
        ensure_ascii=False,
    )

    show_blocked_json = json.dumps(
        bool(show_blocked_relations)
    )

    # ============================================================
    # HTML
    # ============================================================

    html = """
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>Interactive dependence graph</title>

<script
    type="text/javascript"
    src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js">
</script>

<style>

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: white;
}

#controls {
    padding: 10px 14px;
    border-bottom: 1px solid #ddd;
    background: #fafafa;
    display: flex;
    align-items: center;
    flex-wrap: wrap;
    gap: 8px;
}

#network {
    width: 100%;
    height: __HEIGHT__;
}

button {
    padding: 7px 12px;
    border: 1px solid #aaa;
    border-radius: 4px;
    background: white;
    cursor: pointer;
    font-size: 13px;
}

button:hover {
    background: #f0f0f0;
}

.help {
    margin-left: 10px;
    font-size: 13px;
    color: #555;
}

#status {
    margin-left: auto;
    font-size: 12px;
    color: #666;
}

</style>

</head>


<body>


<div id="controls">

    <button id="collapseAll">
        Collapse all
    </button>

    <button id="expandAll">
        Expand all
    </button>

    <button id="fitGraph">
        Fit
    </button>

    <span
        id="helpText"
        class="help">
    </span>

    <span id="status"></span>

</div>


<div id="network"></div>


<script>


const nodes = new vis.DataSet(
    __NODES_JSON__
);


const edges = new vis.DataSet(
    __EDGES_JSON__
);


const childrenBySeparator =
    __CHILDREN_JSON__;


const separatorsByChild =
    __SEPARATORS_BY_CHILD_JSON__;


const collapsibleNodes =
    new Set(
        __COLLAPSIBLE_NODES_JSON__
    );


const pureSeparatorNodes =
    new Set(
        __PURE_SEPARATOR_NODES_JSON__
    );


const showBlockedRelations =
    __SHOW_BLOCKED_RELATIONS__;


const expandedSeparators =
    new Set();


const container =
    document.getElementById(
        "network"
    );


const statusElement =
    document.getElementById(
        "status"
    );


const helpElement =
    document.getElementById(
        "helpText"
    );


const data = {
    nodes: nodes,
    edges: edges
};


const options = {

    layout: {

        hierarchical: {

            enabled: true,

            direction: "LR",

            sortMethod: "directed",

            levelSeparation: 330,

            nodeSpacing: 42,

            treeSpacing: 180,

            blockShifting: true,

            edgeMinimization: true,

            parentCentralization: true
        }
    },

    physics: {
        enabled: false
    },

    interaction: {

        hover: true,

        navigationButtons: true,

        keyboard: true,

        tooltipDelay: 150,

        zoomView: true,

        dragView: true
    },

    nodes: {

        borderWidth: 1,

        font: {
            size: 13,
            face: "Arial"
        }
    },

    groups: {

        output: {
            shape: "ellipse",
            size: 32
        },

        robust: {
            shape: "dot",
            size: 24
        },

        separator: {
            shape: "dot",
            size: 18
        },

        associated: {
            shape: "dot",
            size: 11
        }
    },

    edges: {

        smooth: {

            enabled: true,

            type: "cubicBezier",

            forceDirection: "horizontal",

            roundness: 0.25
        }
    }
};


const network =
    new vis.Network(
        container,
        data,
        options
    );


// ============================================================
// Help
// ============================================================

if (showBlockedRelations) {

    helpElement.textContent =
        "Click visible separator nodes to expand/collapse their parents. "
        + "Solid = robust relation. "
        + "Dashed = relation removed by conditioning.";

} else {

    helpElement.textContent =
        "Robust network with expandable blocking structures. "
        + "Dashed blocked relations are hidden.";
}


// ============================================================
// Child still needed by another expanded separator?
// ============================================================

function childStillNeeded(childId) {

    const parentSeparators =
        separatorsByChild[
            childId
        ] || [];

    for (
        const separatorId
        of parentSeparators
    ) {

        if (
            expandedSeparators.has(
                separatorId
            )
        ) {
            return true;
        }
    }

    return false;
}


// ============================================================
// Show/hide dashed X -> output edge
// ============================================================

function updateBlockedEdge(childId) {

    if (!showBlockedRelations) {
        return;
    }

    const node =
        nodes.get(
            childId
        );

    if (!node) {
        return;
    }

    const blockedEdges =
        edges.get({

            filter: function(edge) {

                return (
                    edge.type === "blocked"
                    &&
                    edge.from === childId
                );
            }
        });

    blockedEdges.forEach(
        edge => {

            edges.update({
                id: edge.id,
                hidden: Boolean(node.hidden)
            });
        }
    );
}


// ============================================================
// Reveal pure separator when required
// ============================================================

function revealSeparator(
    separatorId
) {

    if (
        pureSeparatorNodes.has(
            separatorId
        )
    ) {

        nodes.update({
            id: separatorId,
            hidden: false
        });
    }
}


// ============================================================
// Hide pure separator if it would become orphaned
// ============================================================

function hideSeparatorIfUnused(
    separatorId
) {

    if (
        !pureSeparatorNodes.has(
            separatorId
        )
    ) {
        return;
    }

    if (
        expandedSeparators.has(
            separatorId
        )
    ) {
        return;
    }

    nodes.update({
        id: separatorId,
        hidden: true
    });
}


// ============================================================
// Expand separator
// ============================================================

function expand(
    separatorId
) {

    const children =
        childrenBySeparator[
            separatorId
        ];

    if (!children) {
        return;
    }

    revealSeparator(
        separatorId
    );

    expandedSeparators.add(
        separatorId
    );

    children.forEach(
        childId => {

            if (
                collapsibleNodes.has(
                    childId
                )
            ) {

                nodes.update({
                    id: childId,
                    hidden: false
                });
            }

            const motifEdges =
                edges.get({

                    filter: function(edge) {

                        return (
                            edge.type === "motif"
                            &&
                            edge.from === childId
                            &&
                            edge.to === separatorId
                        );
                    }
                });

            motifEdges.forEach(
                edge => {

                    edges.update({
                        id: edge.id,
                        hidden: false
                    });
                }
            );

            updateBlockedEdge(
                childId
            );
        }
    );

    updateStatus();
}


// ============================================================
// Collapse separator
// ============================================================

function collapse(
    separatorId
) {

    const children =
        childrenBySeparator[
            separatorId
        ];

    if (!children) {
        return;
    }

    expandedSeparators.delete(
        separatorId
    );

    children.forEach(
        childId => {

            const motifEdges =
                edges.get({

                    filter: function(edge) {

                        return (
                            edge.type === "motif"
                            &&
                            edge.from === childId
                            &&
                            edge.to === separatorId
                        );
                    }
                });

            motifEdges.forEach(
                edge => {

                    edges.update({
                        id: edge.id,
                        hidden: true
                    });
                }
            );

            if (
                collapsibleNodes.has(
                    childId
                )
                &&
                !childStillNeeded(
                    childId
                )
            ) {

                nodes.update({
                    id: childId,
                    hidden: true
                });
            }

            updateBlockedEdge(
                childId
            );
        }
    );

    hideSeparatorIfUnused(
        separatorId
    );

    updateStatus();
}


// ============================================================
// Toggle
// ============================================================

function toggleSeparator(
    separatorId
) {

    if (
        expandedSeparators.has(
            separatorId
        )
    ) {

        collapse(
            separatorId
        );

    } else {

        expand(
            separatorId
        );
    }
}


// ============================================================
// Status
// ============================================================

function updateStatus() {

    statusElement.textContent =
        "Expanded separators: "
        + expandedSeparators.size;
}


// ============================================================
// Click
// ============================================================

network.on(

    "click",

    function(params) {

        if (
            params.nodes.length !== 1
        ) {
            return;
        }

        const id =
            params.nodes[0];

        if (
            !Object.prototype
                .hasOwnProperty
                .call(
                    childrenBySeparator,
                    id
                )
        ) {
            return;
        }

        toggleSeparator(
            id
        );
    }
);


// ============================================================
// Collapse all
// ============================================================

document
    .getElementById(
        "collapseAll"
    )
    .addEventListener(

        "click",

        function() {

            const expanded =
                Array.from(
                    expandedSeparators
                );

            expanded.forEach(
                separatorId => {

                    collapse(
                        separatorId
                    );
                }
            );

            network.fit({
                animation: true
            });
        }
    );


// ============================================================
// Expand all
// ============================================================

document
    .getElementById(
        "expandAll"
    )
    .addEventListener(

        "click",

        function() {

            Object.keys(
                childrenBySeparator
            ).forEach(
                separatorId => {

                    expand(
                        separatorId
                    );
                }
            );

            network.fit({
                animation: true
            });
        }
    );


// ============================================================
// Fit
// ============================================================

document
    .getElementById(
        "fitGraph"
    )
    .addEventListener(

        "click",

        function() {

            network.fit({
                animation: true
            });
        }
    );


// ============================================================
// Initial state
// ============================================================

updateStatus();


setTimeout(

    function() {

        network.fit({
            animation: false
        });

    },

    150
);


</script>

</body>

</html>
"""

    # ============================================================
    # Substitute placeholders
    # ============================================================

    html = (
        html
        .replace(
            "__HEIGHT__",
            str(height),
        )
        .replace(
            "__NODES_JSON__",
            nodes_json,
        )
        .replace(
            "__EDGES_JSON__",
            edges_json,
        )
        .replace(
            "__CHILDREN_JSON__",
            children_json,
        )
        .replace(
            "__SEPARATORS_BY_CHILD_JSON__",
            separators_by_child_json,
        )
        .replace(
            "__COLLAPSIBLE_NODES_JSON__",
            collapsible_json,
        )
        .replace(
            "__PURE_SEPARATOR_NODES_JSON__",
            pure_separator_json,
        )
        .replace(
            "__SHOW_BLOCKED_RELATIONS__",
            show_blocked_json,
        )
    )

    # ============================================================
    # Write file
    # ============================================================

    output_file = Path(
        output_file
    )

    output_file.write_text(
        html,
        encoding="utf-8",
    )

    # ============================================================
    # Diagnostics
    # ============================================================

    print(
        "Interactive graph written to:"
    )
    print(
        output_file.resolve()
    )
    print()

    print(
        "Nodes:",
        len(nodes),
    )

    print(
        "Unique node IDs:",
        len(
            {
                node["id"]
                for node in nodes
            }
        ),
    )

    print(
        "Edges:",
        len(edges),
    )

    print(
        "Robust neighbours:",
        len(robust),
    )

    print(
        "Separators:",
        len(separators),
    )

    print(
        "Pure separators initially hidden:",
        len(pure_separator_nodes),
    )

    print(
        "Associated variables:",
        len(associated),
    )

    print(
        "Motifs:",
        len(motifs),
    )

    print(
        "Show blocked relations:",
        show_blocked_relations,
    )

    return output_file

In [12]:
interactive_file = plot_interactive_dependency_graph(
    results_within,
    output_var="MBS_SCT_CD",
    output_label="SCT CD",
    output_file="interactive_dependence_graph_.html",
    height="850px",
    show_blocked_relations=True
)

Interactive graph written to:
C:\workspace\costimiser\causality\interactive_dependence_graph_.html

Nodes: 278
Unique node IDs: 278
Edges: 512
Robust neighbours: 6
Separators: 109
Pure separators initially hidden: 105
Associated variables: 253
Motifs: 253
Show blocked relations: True
